# Aesteel — trening, walidacja i predykcja

Notebook respektuje zasady projektu: XGBoost dla label i severity, Isolation Forest jako niezależny anomaly signal, estymatory fitowane wyłącznie na train.csv, val.csv jako semantic reference, a test.csv tylko do predykcji. Ponieważ val jest potrzebne do pseudo-labelowania, dzielimy je na reference i niewidziany hold-out evaluation. Raw score = 0.75 * Macro-F1(label) + 0.25 * severity accuracy.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
from src.common import clean_spectrum, LABELS, FAULT_LABELS
from src.semi_supervised import SemiSupervisedDiagnosticPipeline

DATA = ROOT / "data"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
print("Project root:", ROOT)


In [ ]:
train = clean_spectrum(pd.read_csv(DATA / "train.csv"))
val = clean_spectrum(pd.read_csv(DATA / "val.csv"))
test = clean_spectrum(pd.read_csv(DATA / "test.csv"))
print("train:", train.shape, "(unlabeled)")
print("val:", val.shape, "(labeled)")
print("test:", test.shape, "(prediction only)")
print("\nVal labels:\n", val["label"].value_counts())
print("\nVal severity:\n", val["severity"].value_counts())


## 1. Hold-out validation

`val_reference` jest używany wyłącznie jako semantic reference bank do pseudo-labelowania train. `val_eval` nie jest przekazywany do fit i służy tylko do oceny.


In [ ]:
val_reference, val_eval = train_test_split(
    val, test_size=0.50, random_state=42, stratify=val["label"]
)
print("reference:", val_reference.shape)
print("evaluation:", val_eval.shape)
print(val_eval["label"].value_counts())


## 2. Feature analysis + trening

Najpierw XGBoost analizuje importance wszystkich cech. Finalny model korzysta z TOP 32. Następnie niezależnie fitowany jest Isolation Forest na wybranych cechach.


In [ ]:
pipeline = SemiSupervisedDiagnosticPipeline(
    n_estimators=360, max_depth=4, learning_rate=0.035,
    subsample=0.85, colsample_bytree=0.80, min_child_weight=5,
    reg_alpha=0.20, reg_lambda=3.0, gamma=0.05, max_bin=128,
    feature_count=32, random_state=42, unknown_confidence=0.45
)
pipeline.fit(train, val_reference)

print("Label model:", type(pipeline.model).__name__)
print("Severity models:", len(pipeline.severity_models))
print("Anomaly:", type(pipeline.isolation_forest).__name__)
print("Features:", len(pipeline.feature_names), "->", len(pipeline.selected_features))
display(pd.DataFrame(pipeline.feature_analysis[:20]))


## 3. Walidacja

Severity jest oceniane z zasadą: `ok` i `unknown` → `nie_dotyczy`; pozostałe cztery klasy → `male/srednie/duze`.


In [ ]:
val_pred = pipeline.predict(val_eval)
y_true_label = val_eval["label"].astype(str).to_numpy()
y_pred_label = val_pred["label"].astype(str).to_numpy()

label_accuracy = accuracy_score(y_true_label, y_pred_label)
label_macro_f1 = f1_score(y_true_label, y_pred_label, labels=LABELS, average="macro", zero_division=0)

y_true_severity = np.where(
    val_eval["label"].astype(str).isin(FAULT_LABELS),
    val_eval["severity"].astype(str), "nie_dotyczy"
)
y_pred_severity = np.where(
    np.isin(y_pred_label, FAULT_LABELS),
    val_pred["severity"].astype(str), "nie_dotyczy"
)

severity_accuracy_all = accuracy_score(y_true_severity, y_pred_severity)
fault_mask = np.isin(y_true_label, FAULT_LABELS)
severity_accuracy_faults = accuracy_score(
    y_true_severity[fault_mask], y_pred_severity[fault_mask]
) if fault_mask.any() else float("nan")

raw_score = 0.75 * label_macro_f1 + 0.25 * severity_accuracy_all

print(f"Label accuracy:             {label_accuracy:.4f}")
print(f"Label Macro-F1:             {label_macro_f1:.4f}")
print(f"Severity accuracy (all):    {severity_accuracy_all:.4f}")
print(f"Severity accuracy (faults): {severity_accuracy_faults:.4f}")
print(f"RAW SCORE:                  {raw_score:.4f}")


In [ ]:
print("LABEL REPORT")
print(classification_report(y_true_label, y_pred_label, labels=LABELS, zero_division=0))
print("LABEL CONFUSION MATRIX")
display(pd.DataFrame(
    confusion_matrix(y_true_label, y_pred_label, labels=LABELS),
    index=LABELS, columns=LABELS
))

print("SEVERITY REPORT")
print(classification_report(
    y_true_severity, y_pred_severity,
    labels=["male", "srednie", "duze", "nie_dotyczy"],
    zero_division=0
))


## 4. Test prediction

`test.csv` nie jest używany do treningu, feature selection ani strojenia. Tutaj wykonujemy wyłącznie inference i zapisujemy predykcje.


In [ ]:
test_pred = pipeline.predict(test)
test_output = test[["engine_id", "cylinder"]].copy()
test_output["label"] = test_pred["label"].astype(str).to_numpy()
test_output["severity"] = test_pred["severity"].astype(str).to_numpy()
test_output["confidence"] = test_pred["confidence"].to_numpy()
test_output["anomaly_score"] = test_pred["anomaly_score"].to_numpy()

test_output.loc[test_output["label"].isin(["ok","unknown"]), "severity"] = "nie_dotyczy"
test_path = OUTPUT / "test_predictions.csv"
test_output.to_csv(test_path, index=False)
display(test_output.head(20))
print("Saved:", test_path)


In [ ]:
model_path = ROOT / "models" / "xgboost_model_notebook.pkl"
pipeline.save(model_path)
assert len(test_output) == len(test)
assert set(test_output.loc[test_output["label"].isin(["ok","unknown"]), "severity"]) <= {"nie_dotyczy"}
assert set(test_output.loc[test_output["label"].isin(FAULT_LABELS), "severity"]) <= {"male","srednie","duze"}
print("All sanity checks passed.")
print(f"RAW SCORE (hold-out val): {raw_score:.4f}")
print("Model saved:", model_path)
